In [12]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "brauer2006making")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Braeuer_2006_causal_ape_exeltabelle.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [13]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)

df['study_id']="brauer2006making"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
df = df.rename(columns={"subject": "ape"})


In [14]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()

for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left') 

In [15]:
# df.columns
df = df.rename(columns={"control.1": "control_1",
    "banane vs. pyramide": "banane_vs_pyramide",
    "solidblock with middle": "solidblock_with_middle",
    'reach':'reach_first',
    'noiseghost':'noise_ghost',
    'tryingopen':'try_to_open',
    'shapeghost':'shape_ghost',
    'pointstatic':'point_continuous',
    'reachnew':'reach',
    'twoshapes':'shape_smell_control',
    'noisearb':'noise_arbitrary_control',
    'lookstatic':'look_continuous',
    'tryingopen2':'try_to_open_2',
    'noiseempty':'noise_empty',
    'solidblock':'shape_block_control'})

df.rename(columns={"ape": "participant"}, inplace=True)

In [16]:
session_a = df[['study_id','participant','sex', 'species','point','reach_first','shape','noise_ghost']]
session_a = session_a.assign(session='a')
session_b = df[['study_id','participant','sex', 'species','look', 'try_to_open', 'noise','shape_ghost']]
session_b = session_b.assign(session='b')
session_c = df[['study_id','participant','sex', 'species', 'point_continuous','reach', 'shape_smell_control', 'noise_arbitrary_control']]
session_c = session_c.assign(session='c')
session_d = df[['study_id','participant','sex', 'species', 'look_continuous', 'try_to_open_2', 'noise_empty', 'shape_block_control']]
session_d = session_d.assign(session='d')


In [17]:
control_temp = df[['study_id','participant','sex', 'species', 'control']].values.tolist() + df[[
              'study_id','participant','sex', 'species', 'control_1']].values.tolist()

session_control = pd.DataFrame(control_temp, columns=['study_id','participant','sex', 'species', 
    'no_cue_control'])
session_control=session_control.sort_values(by = ['participant'])
session_control.dropna(subset=['no_cue_control'], inplace=True)
session_control = session_control.assign(session='control')

In [18]:
data_frames = [session_a, session_b, session_c, session_d, session_control]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)


In [19]:
complete_path_age = os.path.join(original_data_pathway, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
fulldf= fulldf.merge(subject_list,left_on='participant', right_on='name', how='left')
fulldf.rename(columns={"age": "age_in_years"}, inplace=True)

In [20]:

brauer2006making_standardized= fulldf[['study_id', 'participant','age_in_years',  'sex', 'species', 'session','point', 'reach_first',
       'shape', 'noise_ghost', 'look', 'try_to_open', 'noise',
       'shape_ghost', 'point_continuous', 'reach', 'shape_smell_control',
       'noise_arbitrary_control', 'look_continuous', 'try_to_open_2',
       'noise_empty', 'shape_block_control', 'no_cue_control']]
# 
# df[['study_id','participant', 
#         'sex', 'species', 'look', 'point', 'reach', 'tryingopen', 'shape', 'shapeghost',
#        'noise', 'noiseghost', 'control', 'lookstatic', 'pointstatic',
#        'reachnew', 'tryingopen2', 'twoshapes', 'solidblock', 'noisearb',
#        'noiseempty', 'control_1', 'banane', 'banane_vs_pyramide',
#        'solidblock_with_middle' ]]




In [21]:
comp_out_path_stand = os.path.join(out_pathway, 'brauer2006making_standardized.csv')
brauer2006making_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

In [22]:

names =brauer2006making_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
brauer2006making_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'brauer2006making_glossary.csv')
brauer2006making_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
